In [ ]:
# =======================================================
# CELL 1 - GPU CHECK + DRIVE MOUNT
# Run first. If GPU is missing, stop here and change runtime.
# =======================================================
import os, sys, subprocess
from pathlib import Path
import torch
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"OK GPU: {name} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError(
        "\nNo GPU detected.\n"
        "Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU\n"
        "Then re-run this cell."
    )

print("OK Drive mounted.")

In [ ]:
# =======================================================
# CELL 2 - INSTALL ALL DEPENDENCIES
# Safe to re-run. Takes a few minutes on first run.
# =======================================================
import sys, subprocess

packages = [
    "torch", "numpy", "PyYAML", "pytest", "tqdm",
    "sympy", "scipy", "psutil", "networkx",
    "fastapi", "uvicorn", "httpx",
    "sentence-transformers",
    "transformers", "accelerate",
]

print("Installing standard packages...")
for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    icon = "OK" if r.returncode == 0 else "WARN"
    print(f"  {icon} {pkg}")

print("\nInstalling Muon optimizer from GitHub...")
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/KellerJordan/Muon"],
    capture_output=True, text=True
)
if r.returncode == 0:
    print("  OK muon")
else:
    print("  WARN muon install failed - AdamW fallback will be used automatically")

print("\nOK All dependencies installed.")

In [ ]:
# =======================================================
# CELL 3 - CLONE OR UPDATE AN-RA
# Clones fresh if not present. Pulls latest if already there.
# =======================================================
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"

if not REPO.exists():
    print("Cloning An-Ra (first time)...")
    r = subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"Clone failed:\n{r.stderr}")
    print("OK Cloned.")
else:
    r = subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], capture_output=True, text=True)
    print(f"OK {r.stdout.strip() or 'Already up to date.'}")

os.environ["ANRA_REPO_ROOT"] = str(REPO)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from anra_paths import inject_all_paths, ensure_dirs
inject_all_paths()
ensure_dirs()
print(f"OK Paths injected. Working dir: {REPO}")

In [ ]:
# =======================================================
# CELL 4 - RESTORE ARTIFACTS
# Uses repo files first, restores from Drive if available.
# =======================================================
import os, sys, shutil, pickle
from pathlib import Path

REPO = Path("/content/An-Ra")
DRIVE = Path("/content/drive/MyDrive/AnRa")

if not REPO.exists():
    raise RuntimeError(
        f"Repo not found at {REPO}.\n"
        "Run the clone/update cell first."
    )

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

os.chdir(REPO)

from anra_paths import inject_all_paths, ensure_dirs, TOKENIZER_DIR, TRAINING_DATA_DIR
inject_all_paths()
ensure_dirs()

print("=== Restoring An-Ra artifacts from Drive ===\n")

for name in ["anra_brain.pt", "anra_brain_identity.pt", "anra_ouroboros.pt"]:
    local_path = REPO / name
    drive_path = DRIVE / "checkpoints" / name
    if local_path.exists() and local_path.stat().st_size > 10_000:
        print(f"  OK {name} already present in repo ({local_path.stat().st_size / 1e6:.1f} MB)")
    elif drive_path.exists() and drive_path.stat().st_size > 10_000:
        shutil.copy2(drive_path, local_path)
        print(f"  OK {name} restored from Drive ({local_path.stat().st_size / 1e6:.1f} MB) -> will resume training")
    else:
        print(f"  INFO {name}: not found locally or on Drive -> fresh start")

TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
tok_local = TOKENIZER_DIR / "tokenizer.pkl"
tok_drive = DRIVE / "tokenizer.pkl"

if tok_local.exists() and tok_local.stat().st_size > 100:
    with open(tok_local, "rb") as f:
        tok = pickle.load(f)
    print(f"\n  OK tokenizer.pkl already present in repo (vocab_size={tok.vocab_size})")
elif tok_drive.exists() and tok_drive.stat().st_size > 100:
    shutil.copy2(tok_drive, tok_local)
    with open(tok_local, "rb") as f:
        tok = pickle.load(f)
    print(f"\n  OK tokenizer.pkl restored from Drive (vocab_size={tok.vocab_size})")
else:
    print("\n  INFO tokenizer.pkl not found locally or on Drive -> build_brain.py will auto-create it")

ds_local = TRAINING_DATA_DIR / "anra_dataset_v6_1.txt"
ds_drive = DRIVE / "anra_dataset_v6_1.txt"

if ds_local.exists() and ds_local.stat().st_size > 100_000:
    print(f"\n  OK Dataset already present in repo: {ds_local} ({ds_local.stat().st_size / 1e6:.2f} MB)")
elif ds_drive.exists() and ds_drive.stat().st_size > 100_000:
    TRAINING_DATA_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ds_drive, ds_local)
    print(f"\n  OK Dataset restored from Drive: {ds_local} ({ds_local.stat().st_size / 1e6:.2f} MB)")
else:
    print(
        f"\n  WARN Dataset not found locally or on Drive.\n"
        f"    Expected local: {ds_local}\n"
        f"    Expected drive: {ds_drive}\n"
        f"    Training cannot start until the dataset exists."
    )

print("\nOK Artifact restoration complete.")

In [ ]:
# =======================================================
# CELL 5 - SYSTEM HEALTH CHECK
# Verify every module before training starts.
# =======================================================
import os, sys, importlib, pickle, subprocess, torch
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths, ensure_dirs, TRAINING_DATA_DIR, TOKENIZER_DIR
inject_all_paths()
ensure_dirs()

print("================ AN-RA FULL SYSTEM HEALTH CHECK ================\n")

print("-- Critical Files --")
critical = {
    "anra_brain.py": REPO / "anra_brain.py",
    "anra_paths.py": REPO / "anra_paths.py",
    "build_brain.py": REPO / "scripts" / "build_brain.py",
    "train_unified.py": REPO / "training" / "train_unified.py",
    "Training dataset": TRAINING_DATA_DIR / "anra_dataset_v6_1.txt",
}
for label, path in critical.items():
    if path.exists():
        size = f" ({path.stat().st_size / 1e6:.2f} MB)" if path.suffix in {".py", ".txt"} else ""
        print(f"  OK {label}{size}")
    else:
        print(f"  FAIL {label}: MISSING - {path}")

print("\n-- Tokenizer --")
tok_path = TOKENIZER_DIR / "tokenizer.pkl"
if tok_path.exists():
    with open(tok_path, "rb") as f:
        tok = pickle.load(f)
    print(f"  OK tokenizer.pkl vocab_size={tok.vocab_size}")
else:
    print("  INFO tokenizer.pkl: will be auto-built at training start")

print("\n-- Checkpoint --")
ckpt = REPO / "anra_brain.pt"
if ckpt.exists():
    try:
        state = torch.load(ckpt, map_location="cpu", weights_only=False)
        step = state.get("global_step", "?") if isinstance(state, dict) else "?"
        loss = state.get("best_loss", "?") if isinstance(state, dict) else "?"
        print(f"  OK anra_brain.pt step={step} best_loss={loss}")
    except Exception as e:
        print(f"  WARN anra_brain.pt: corrupted ({e}) - will train fresh")
else:
    print("  INFO anra_brain.pt: not found - will train from scratch")

print("\n-- Module Health --")
all_modules = {
    "TurboQuant         ": "turboquant",
    "Identity    (45N)  ": "identity_injector",
    "Ouroboros   (45O)  ": "ouroboros_numpy",
    "Ghost Mem   (45P)  ": "ghost_memory",
    "Symbolic    (45Q)  ": "symbolic_bridge",
    "Sovereignty (45R)  ": "sovereignty_bridge",
    "Memory      (45J)  ": "memory_manager",
    "Agent Loop  (45K)  ": "agent_main",
    "Self-Impr   (45L)  ": "improve",
    "Master Sys  (45M)  ": "system",
}
for label, mod_name in all_modules.items():
    try:
        mod = importlib.import_module(mod_name)
        fn = getattr(mod, "health_check", None)
        if callable(fn):
            result = fn()
            status = result.get("status", "ok") if isinstance(result, dict) else "ok"
            detail = result.get("reason", "") if isinstance(result, dict) and status != "ok" else ""
            suffix = f" - {detail}" if detail else ""
            print(f"  {'OK' if status == 'ok' else 'WARN'} {label}: {status}{suffix}")
        else:
            print(f"  OK {label}: importable")
    except Exception as e:
        print(f"  WARN {label}: {type(e).__name__}: {e}")

print("\n-- GPU --")
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  OK {torch.cuda.get_device_name(0)} {used:.1f}/{total:.1f} GB")
else:
    print("  FAIL GPU not available - go to Runtime -> Change runtime type -> T4 GPU")

print("\nOK Health check complete. If dataset is present, proceed to Cell 6.")

In [ ]:
# =======================================================
# CELL 6 - TRAIN AN-RA
# 30-minute session. Auto-resumes from checkpoint.
# Saves only at the end - no interruptions during training.
# =======================================================
import os, sys, subprocess, json
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths, TRAINING_DATA_DIR
inject_all_paths()

dataset_path = TRAINING_DATA_DIR / "anra_dataset_v6_1.txt"
if not dataset_path.exists() or dataset_path.stat().st_size < 100_000:
    raise RuntimeError(
        f"Training dataset missing or too small: {dataset_path}\n"
        "Run Cell 4 first and make sure anra_dataset_v6_1.txt exists in the repo or on Drive."
    )

print("========================================================")
print("AN-RA TRAINING SESSION")
print("30 min | auto-resume | saves once at end to Drive")
print("========================================================\n")

cmd = [
    sys.executable, "-m", "training.train_unified",
    "--mode", "train",
    "--session_minutes", "30",
    "--checkpoint_path", "anra_brain.pt",
    "--block_size", "256",
    "--batch_size", "64",
    "--ouroboros_steps", "5000",
]

print("Command:", " ".join(cmd))
print("-" * 58)

proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=str(REPO),
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()

print("-" * 58)

if proc.returncode == 0:
    print("\nOK Training complete. Checkpoint saved to Drive.")
else:
    raise RuntimeError(f"Training exited with code {proc.returncode}. Read the log above.")

for rname in ["unified_training_report.json", "session_train_metrics.json", "next_session_curriculum.json"]:
    rp = REPO / "output" / rname
    if rp.exists():
        try:
            report = json.loads(rp.read_text(encoding="utf-8"))
            print(f"\n-- {rname} --")
            for key, value in report.items():
                if key == "stages":
                    print(f"  stages: {value}")
                else:
                    print(f"  {key}: {value}")
        except Exception:
            pass

drive_ckpt = Path("/content/drive/MyDrive/AnRa/checkpoints/anra_brain.pt")
if drive_ckpt.exists():
    print(f"\nOK Drive backup: {drive_ckpt.stat().st_size / 1e6:.1f} MB")
    print("Come back anytime - Cell 6 will resume exactly where this stopped.")
else:
    print(f"\nWARN Drive backup not found at expected path: {drive_ckpt}")

In [ ]:
# =======================================================
# CELL 7 - POST-TRAINING VERIFICATION
# Run after Cell 6 to confirm everything saved correctly.
# =======================================================
import os, sys, subprocess
from pathlib import Path
from google.colab import drive

REPO = Path("/content/An-Ra")
GITHUB_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"
if not Path("/content/drive/MyDrive").exists():
    drive.mount('/content/drive', force_remount=False)
if not REPO.exists():
    subprocess.run(["git", "clone", GITHUB_URL, str(REPO)], check=True)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.chdir(REPO)

from anra_paths import inject_all_paths
inject_all_paths()

print("=== Post-Training Verification ===\n")

r = subprocess.run([sys.executable, "scripts/verify_structure.py"], capture_output=True, text=True, cwd=str(REPO))
print("-- Structure --")
print(r.stdout.strip() or r.stderr.strip())

r = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--tb=short", "--no-header"], capture_output=True, text=True, cwd=str(REPO))
print("\n-- Tests --")
out = (r.stdout + r.stderr).strip()
print(out[-3000:] if len(out) > 3000 else out)

print("\n-- Drive Backup --")
drive = Path("/content/drive/MyDrive/AnRa")
for name in [
    "checkpoints/anra_brain.pt",
    "checkpoints/anra_brain_identity.pt",
    "checkpoints/anra_ouroboros.pt",
    "tokenizer.pkl",
    "anra_dataset_v6_1.txt",
]:
    p = drive / name
    if p.exists():
        print(f"  OK {name} ({p.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"  INFO {name}: not on Drive")

print("\nOK Done.")